# 01c Parse MEPS Ensemble Forecast Uncertainty Features

This notebook parses archived MEPS ensemble NetCDF files into forecast-uncertainty summaries aligned to the Europe/Oslo local-time windows used in notebooks 01a and 01b: `full_day_00_23`, `trade_08_18`, and `trade_08_19`. The outputs support ensemble-spread diagnostics, M4 uncertainty features, and later forecast-error calibration.

Full parsing should only be enabled when the ensemble folders are sufficiently complete. `ensemble_wind_y_nc` is required together with `ensemble_wind_x_nc` before wind speed can be computed, because `wind_x` alone is not scalar wind speed. Final processed outputs are written only when `RUN_FULL_PARSE = True` and the save gate is satisfied.


## Imports and constants

The setup defines project-root discovery, ensemble input folders, output paths, variable mappings, horizon/window settings, and parse gates that prevent accidental full execution.


In [ ]:
import gc
import os
import re
import time
from datetime import datetime, time as dt_time, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import netCDF4
except ImportError as exc:
    raise ImportError("netCDF4 is required in the existing csvi_env kernel.") from exc


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start] + list(start.parents):
        if any((candidate / marker).exists() for marker in ["README_FOR_CODEX.md", "AGENTS.md", ".git"]):
            return candidate
    raise FileNotFoundError("Could not locate project root from marker files.")


PROJECT_ROOT = find_project_root()
RAW_DATA_ROOT = Path(os.environ.get("FIMEX_RAW_ROOT", Path.home() / "fimex_out")).resolve()
DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
STORE_MAPPING_PATH = PROJECT_ROOT / "thesis_context" / "Avd_lokasjon.xlsx"
CLOUD_POINTS_CFG = RAW_DATA_ROOT / "cloud_points.cfg"

REALISED_WINDOWS_PATH = PROCESSED_DIR / "realised_weather_daily_windows.parquet"
DETERMINISTIC_WINDOWS_PATH = PROCESSED_DIR / "forecast_meps_daily_windows.parquet"
HORIZON_HOUR_AUDIT_PATH = PROCESSED_DIR / "forecast_meps_horizon_hour_audit.parquet"
ENSEMBLE_WINDOWS_OUTPUT_PATH = PROCESSED_DIR / "forecast_meps_ensemble_daily_windows.parquet"
ENSEMBLE_COVERAGE_AUDIT_OUTPUT_PATH = PROCESSED_DIR / "forecast_meps_ensemble_coverage_audit.parquet"
OPTIONAL_CALIBRATION_OUTPUT_PATH = PROCESSED_DIR / "forecast_error_calibration_windows.parquet"

LOCAL_TZ = ZoneInfo("Europe/Oslo")
UTC_TZ = ZoneInfo("UTC")
SENTINEL_ABS_THRESHOLD = 1e6
WET_THRESHOLD_MM = 0.1
# Accumulated ensemble precipitation can contain tiny (<0.01 mm) packing artefacts;
# larger negative drops remain invalid and are audited.
PRECIP_TINY_NEGATIVE_TOLERANCE_MM = 1e-2
PRECIP_LARGE_NEGATIVE_TOLERANCE_MM = PRECIP_TINY_NEGATIVE_TOLERANCE_MM
PRECIP_RESET_DROP_THRESHOLD_MM = 5e-2
PRECIP_NEGATIVE_TOLERANCE = PRECIP_TINY_NEGATIVE_TOLERANCE_MM  # Backward-compatible alias.
PRECIP_AUDIT_COLUMNS = [
    "negative_diff_count", "tiny_negative_count", "large_negative_count",
    "reset_like_count", "reset_handling_count", "negative_diff_min",
    "negative_diff_p01", "negative_diff_p50", "negative_diff_p99",
    "precip_tiny_negative_tolerance_mm", "precip_large_negative_policy",
]
LAT_LON_TOLERANCE = 1e-3

RUN_SMOKE_TEST = True
SMOKE_FILES_PER_VARIABLE = 1
RUN_FULL_PARSE = True
ALLOW_PARTIAL_FINAL_OUTPUT = False

WINDOW_DEFINITIONS = {
    "full_day_00_23": {"start_hour": 0, "end_hour": 23},
    "trade_08_18": {"start_hour": 8, "end_hour": 18, "expected_hours": 11},
    "trade_08_19": {"start_hour": 8, "end_hour": 19, "expected_hours": 12},
}
WINDOW_ORDER = ["full_day_00_23", "trade_08_18", "trade_08_19"]
WINDOW_OUTPUT_HORIZONS = [0, 1, 2]

ENSEMBLE_COMPONENTS = {
    "temp": {
        "folder": "ensemble_temp_nc",
        "suffix": "temp",
        "ncvar": "air_temperature_2m",
        "prefix": "temp",
    },
    "precip": {
        "folder": "ensemble_precip_nc",
        "suffix": "precip",
        "ncvar": "precipitation_amount_acc",
        "prefix": "precip",
    },
    "wind_x": {
        "folder": "ensemble_wind_x_nc",
        "suffix": "wind_x",
        "ncvar": "x_wind_10m",
        "prefix": "wind_x",
    },
    "wind_y": {
        "folder": "ensemble_wind_y_nc",
        "suffix": "wind_y",
        "ncvar": "y_wind_10m",
        "prefix": "wind_y",
    },
    "humid": {
        "folder": "ensemble_humid_nc",
        "suffix": "humid",
        "ncvar": "relative_humidity_2m",
        "prefix": "humid",
    },
    "cloud": {
        "folder": "ensemble_cloud_nc",
        "suffix": "cloud",
        "ncvar": "cloud_area_fraction",
        "prefix": "cloud",
    },
}
SCALAR_VARIABLES = ["temp", "precip", "humid", "cloud"]
FINAL_VARIABLES = ["temp", "precip", "wind", "humid", "cloud"]
FILENAME_RE = re.compile(r"^ensemble_(\d{8})T(\d{2})Z_(temp|precip|wind_x|wind_y|humid|cloud)\.nc$")
FINAL_KEY_COLS = [
    "origin_date", "origin_hour", "origin_datetime_utc", "target_date",
    "horizon_days", "AvdelingID", "aggregation_window",
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw FIMEX root: {RAW_DATA_ROOT}")
print(f"RUN_FULL_PARSE = {RUN_FULL_PARSE}")


## Path and availability audit

This read-only audit inspects expected ensemble folders and reports file counts, filename parse success, date ranges, origin-hour distributions, and approximate completeness. Missing or incomplete folders are reported rather than imputed.


In [ ]:
def parse_ensemble_filename(path: Path) -> dict | None:
    match = FILENAME_RE.match(path.name)
    if not match:
        return None
    date_raw, hour_raw, suffix = match.groups()
    origin_dt = pd.Timestamp(f"{date_raw[:4]}-{date_raw[4:6]}-{date_raw[6:]} {hour_raw}:00:00", tz="UTC")
    return {
        "file_name": path.name,
        "path": str(path),
        "suffix": suffix,
        "origin_date": origin_dt.tz_localize(None).normalize(),
        "origin_hour": np.int8(int(hour_raw)),
        "origin_datetime_utc": origin_dt.tz_localize(None),
    }


def audit_ensemble_folders() -> pd.DataFrame:
    rows = []
    for component, cfg in ENSEMBLE_COMPONENTS.items():
        folder = RAW_DATA_ROOT / cfg["folder"]
        files = sorted(folder.glob("*.nc")) if folder.exists() else []
        parsed = [parse_ensemble_filename(path) for path in files]
        parsed = [row for row in parsed if row is not None]
        dates = [row["origin_date"] for row in parsed]
        hours = (
            pd.Series([int(row["origin_hour"]) for row in parsed], dtype="int64")
            if parsed else pd.Series(dtype="int64")
        )
        rows.append(
            {
                "component": component,
                "folder": cfg["folder"],
                "path": str(folder),
                "exists": folder.exists(),
                "file_count": len(files),
                "parsed_file_count": len(parsed),
                "bad_filename_count": len(files) - len(parsed),
                "first_origin_date": min(dates) if dates else pd.NaT,
                "last_origin_date": max(dates) if dates else pd.NaT,
                "origin_hour_distribution": (
                    dict(sorted(hours.value_counts().to_dict().items())) if parsed else {}
                ),
            }
        )
    audit = pd.DataFrame(rows)
    max_count = int(audit["file_count"].max()) if not audit.empty else 0
    audit["share_of_largest_folder"] = np.where(max_count > 0, audit["file_count"] / max_count, np.nan)
    return audit


folder_audit = audit_ensemble_folders()
display(folder_audit)

wind_y_count = int(folder_audit.loc[folder_audit["component"] == "wind_y", "file_count"].iloc[0])
if wind_y_count == 0:
    print(
        "WARNING: ensemble_wind_y_nc has no files. Wind speed ensemble parsing "
        "is unavailable until wind_y exists."
    )
else:
    print(f"wind_y files available: {wind_y_count}")


## Store mapping

The next step loads the store-location mapping and `cloud_points.cfg`, then validates the point order needed to map NetCDF `x` indices to `AvdelingID`.


In [ ]:
def parse_cloud_points_cfg(cfg_path: Path) -> tuple[np.ndarray, np.ndarray]:
    text = cfg_path.read_text(encoding="utf-8", errors="replace")
    lat_match = re.search(r"^\s*latitudeValues\s*=\s*([^\n]+)$", text, flags=re.MULTILINE)
    lon_match = re.search(r"^\s*longitudeValues\s*=\s*([^\n]+)$", text, flags=re.MULTILINE)
    if not lat_match or not lon_match:
        raise ValueError(f"Could not locate latitudeValues / longitudeValues in {cfg_path}")
    lats = np.array([float(x.strip()) for x in lat_match.group(1).split(",") if x.strip()], dtype="float64")
    lons = np.array([float(x.strip()) for x in lon_match.group(1).split(",") if x.strip()], dtype="float64")
    return lats, lons


def find_column(frame: pd.DataFrame, candidates: list[str]) -> str:
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    raise KeyError(f"None of {candidates} found in columns {list(frame.columns)}")


if not STORE_MAPPING_PATH.exists():
    raise FileNotFoundError(f"Store mapping missing: {STORE_MAPPING_PATH}")
if not CLOUD_POINTS_CFG.exists():
    raise FileNotFoundError(f"cloud_points.cfg missing: {CLOUD_POINTS_CFG}")

cfg_lats, cfg_lons = parse_cloud_points_cfg(CLOUD_POINTS_CFG)
avd_df = pd.read_excel(STORE_MAPPING_PATH)
col_avd = find_column(avd_df, ["AvdelingID", "Avdeling", "id"])
col_lat = find_column(avd_df, ["Lat", "latitude", "Latitude", "lat"])
col_lon = find_column(avd_df, ["Lon", "longitude", "Longitude", "lon", "Lng"])

if len(avd_df) != len(cfg_lats) or len(cfg_lats) != len(cfg_lons):
    raise ValueError(
        f"Store/grid length mismatch: Excel rows={len(avd_df)}, "
        f"cfg lat={len(cfg_lats)}, cfg lon={len(cfg_lons)}"
    )

max_lat_diff = float(np.nanmax(np.abs(avd_df[col_lat].to_numpy(dtype="float64") - cfg_lats)))
max_lon_diff = float(np.nanmax(np.abs(avd_df[col_lon].to_numpy(dtype="float64") - cfg_lons)))
if max_lat_diff > LAT_LON_TOLERANCE or max_lon_diff > LAT_LON_TOLERANCE:
    raise ValueError(
        f"Store-grid order mismatch: max lat diff={max_lat_diff:.6f}, "
        f"max lon diff={max_lon_diff:.6f}"
    )

STORE_MAPPING = avd_df[[col_avd, col_lat, col_lon]].rename(
    columns={col_avd: "AvdelingID", col_lat: "latitude", col_lon: "longitude"}
)
STORE_MAPPING["AvdelingID"] = STORE_MAPPING["AvdelingID"].astype("int64")
STORE_MAPPING["grid_x_position"] = np.arange(len(STORE_MAPPING), dtype="int16")
AVDELING_BY_X = STORE_MAPPING["AvdelingID"].to_numpy(dtype="int64")
if len(AVDELING_BY_X) != 46:
    raise ValueError(f"Expected 46 stores/grid points; found {len(AVDELING_BY_X)}")

print(f"Store mapping validated: {len(STORE_MAPPING)} stores")
print(f"Max lat diff={max_lat_diff:.6f}, max lon diff={max_lon_diff:.6f}")
display(STORE_MAPPING.head())


## Ensemble file index

The file index parses ensemble filenames by variable, origin date, and origin hour. It defines the available forecast-origin coverage before any NetCDF values are read.


In [ ]:
def build_component_file_index(component: str) -> pd.DataFrame:
    cfg = ENSEMBLE_COMPONENTS[component]
    folder = RAW_DATA_ROOT / cfg["folder"]
    if not folder.exists():
        return pd.DataFrame(
            columns=[
                "component", "file_name", "path", "origin_date", "origin_hour",
                "origin_datetime_utc",
            ]
        )
    rows = []
    for path in sorted(folder.glob("*.nc")):
        parsed = parse_ensemble_filename(path)
        if parsed is None or parsed["suffix"] != cfg["suffix"]:
            continue
        parsed["component"] = component
        rows.append(parsed)
    out = pd.DataFrame(rows)
    return (
        out.sort_values(["origin_datetime_utc", "file_name"]).reset_index(drop=True)
        if not out.empty else out
    )


file_indexes = {component: build_component_file_index(component) for component in ENSEMBLE_COMPONENTS}
for component, index in file_indexes.items():
    if index.empty:
        print(f"{component}: 0 indexed files")
    else:
        print(
            f"{component}: {len(index)} files, {index['origin_date'].min().date()} "
            f"to {index['origin_date'].max().date()}, hours "
            f"{sorted(index['origin_hour'].astype(int).unique().tolist())}"
        )

if not file_indexes["wind_x"].empty and not file_indexes["wind_y"].empty:
    wind_pairs = file_indexes["wind_x"].merge(
        file_indexes["wind_y"],
        on=["origin_date", "origin_hour", "origin_datetime_utc"],
        how="inner",
        suffixes=("_x", "_y"),
    )
else:
    wind_pairs = pd.DataFrame()

print(f"Aligned wind_x/wind_y origin pairs: {len(wind_pairs)}")


## Parser helpers

These helpers compute local-time aggregation windows, convert units, handle accumulated precipitation, pair wind components, and convert ensemble members into per-store window summaries.


In [ ]:
def compute_expected_hours_for_window(local_date, aggregation_window: str, tz: ZoneInfo = LOCAL_TZ) -> int:
    if aggregation_window == "trade_08_18":
        return 11
    if aggregation_window == "trade_08_19":
        return 12
    if aggregation_window != "full_day_00_23":
        raise ValueError(f"Unknown aggregation_window: {aggregation_window}")
    local_day = pd.Timestamp(local_date).date()
    start_local = datetime.combine(local_day, dt_time(0, 0), tzinfo=tz)
    end_local = datetime.combine(local_day + timedelta(days=1), dt_time(0, 0), tzinfo=tz)
    return int((end_local.astimezone(UTC_TZ) - start_local.astimezone(UTC_TZ)).total_seconds() // 3600)


def assign_expected_hours_for_windows(frame: pd.DataFrame, date_col: str = "target_date") -> pd.Series:
    return frame.apply(
        lambda row: compute_expected_hours_for_window(
            row[date_col], row["aggregation_window"]
        ),
        axis=1,
    ).astype("int8")




class NetCDFParseError(ValueError):
    """Controlled skip-worthy NetCDF parse error for audit logging."""

    def __init__(self, status: str, message: str, *, error_type: str | None = None):
        super().__init__(message)
        self.status = status
        self.error_type = error_type or status


def raise_parse_error(status: str, message: str, *, error_type: str | None = None):
    raise NetCDFParseError(status, message, error_type=error_type)


def _series_value(row: pd.Series | None, key: str, default=None):
    if row is None:
        return default
    try:
        return row[key] if key in row.index else default
    except Exception:
        return default


def source_folder_for_variable(variable: str) -> str:
    if variable == "wind":
        return f"{ENSEMBLE_COMPONENTS['wind_x']['folder']} + {ENSEMBLE_COMPONENTS['wind_y']['folder']}"
    if variable in ENSEMBLE_COMPONENTS:
        return ENSEMBLE_COMPONENTS[variable]["folder"]
    return ""


def build_file_parse_audit_row(
    row: pd.Series | None,
    variable: str,
    *,
    status: str,
    error_type: str | None = None,
    error_message: str | None = None,
    n_rows_available: int = 0,
    coverage_share: float | None = None,
    notes: str = "",
    extra_fields: dict | None = None,
) -> pd.DataFrame:
    path_value = _series_value(row, "path")
    if variable == "wind":
        path_x = _series_value(row, "path_x")
        path_y = _series_value(row, "path_y")
        path_value = " | ".join([str(x) for x in [path_x, path_y] if x])
    is_ok = status == "parsed_ok"
    audit_row = {
        "audit_level": "file_parse",
        "variable": variable,
        "source_folder": source_folder_for_variable(variable),
        "path": path_value,
        "origin_date": _series_value(row, "origin_date", pd.NaT),
        "origin_hour": _series_value(row, "origin_hour", pd.NA),
        "origin_datetime_utc": _series_value(row, "origin_datetime_utc", pd.NaT),
        "aggregation_window": pd.NA,
        "horizon_days": pd.NA,
        "n_rows_available": int(n_rows_available),
        "coverage_share": (
            np.float32(coverage_share)
            if coverage_share is not None and pd.notna(coverage_share) else np.nan
        ),
        "is_available": int(is_ok and n_rows_available > 0),
        "is_partial": int((not is_ok) or n_rows_available == 0),
        "status": status,
        "error_type": error_type or ("" if is_ok else status),
        "error_message": error_message or "",
        "notes": notes or ("parsed successfully" if is_ok else "file skipped"),
    }
    if extra_fields:
        audit_row.update(extra_fields)
    return pd.DataFrame([audit_row])


def build_file_success_audit_row(
    row: pd.Series,
    variable: str,
    parsed: pd.DataFrame,
    coverage: pd.DataFrame,
) -> pd.DataFrame:
    mean_coverage = None
    extra_fields = {}
    if coverage is not None and not coverage.empty and "coverage_share" in coverage.columns:
        mean_coverage = float(pd.to_numeric(coverage["coverage_share"], errors="coerce").mean())
    if variable == "precip" and coverage is not None and not coverage.empty:
        for col in PRECIP_AUDIT_COLUMNS:
            if col in coverage.columns:
                vals = coverage[col].dropna()
                extra_fields[col] = vals.iloc[0] if not vals.empty else np.nan
    return build_file_parse_audit_row(
        row,
        variable,
        status="parsed_ok",
        n_rows_available=len(parsed) if parsed is not None else 0,
        coverage_share=mean_coverage,
        notes="file parsed; window rows are represented in window_coverage audit summaries",
        extra_fields=extra_fields,
    )

def mask_fill_values(raw) -> np.ndarray:
    if hasattr(raw, "filled"):
        raw = raw.filled(np.nan)
    arr = np.asarray(raw, dtype="float32")
    return np.where(np.abs(arr) > SENTINEL_ABS_THRESHOLD, np.nan, arr).astype("float32")


def convert_temperature(arr: np.ndarray, units: str | None) -> np.ndarray:
    vals = arr[np.isfinite(arr)]
    units_l = (units or "").lower().strip()
    if units_l.startswith("k") or (vals.size and float(np.nanmedian(vals)) > 100.0):
        arr = arr - np.float32(273.15)
    return arr.astype("float32")


def convert_fraction_to_percent(arr: np.ndarray, units: str | None) -> np.ndarray:
    vals = arr[np.isfinite(arr)]
    if vals.size == 0:
        raise_parse_error(
            "all_missing_values",
            "No finite humidity/cloud values available before percent conversion.",
        )
    units_l = (units or "").lower().strip()
    if units_l in {"", "1", "fraction"} or float(np.nanmax(vals)) <= 1.5:
        arr = arr * np.float32(100.0)
    return np.clip(arr, 0.0, 100.0).astype("float32")


def convert_precip_accumulated_to_hourly(acc: np.ndarray) -> tuple[np.ndarray, dict]:
    if acc is None:
        raise_parse_error("empty_variable", "Precipitation array is None.")
    acc = np.asarray(acc, dtype="float32")
    if acc.ndim != 3:
        raise_parse_error(
            "invalid_shape",
            f"Expected precipitation array as (time, member, store); got shape {acc.shape}.",
        )
    if acc.shape[0] == 0:
        raise_parse_error("empty_variable", f"Precipitation has zero-length time axis: shape {acc.shape}.")
    if acc.shape[1] == 0 or acc.shape[2] == 0:
        raise_parse_error("invalid_shape", f"Precipitation has empty member/store axis: shape {acc.shape}.")
    if not np.isfinite(acc).any():
        raise_parse_error(
            "all_missing_values",
            "Precipitation array contains no finite usable values after fill-value masking.",
        )

    hourly = np.diff(acc, axis=0, prepend=acc[0:1]).astype("float32")
    hourly[0, ...] = 0.0
    finite_hourly = np.isfinite(hourly)
    negative = finite_hourly & (hourly < 0.0)
    tiny_negative = negative & (hourly >= -PRECIP_TINY_NEGATIVE_TOLERANCE_MM)
    large_negative = negative & (hourly < -PRECIP_TINY_NEGATIVE_TOLERANCE_MM)

    neg_values = hourly[negative].astype("float64")
    diag = {
        "negative_diff_count": int(np.count_nonzero(negative)),
        "tiny_negative_count": int(np.count_nonzero(tiny_negative)),
        "large_negative_count": int(np.count_nonzero(large_negative)),
        "reset_like_count": 0,
        "reset_handling_count": 0,
        "negative_diff_min": float(np.nanmin(neg_values)) if neg_values.size else np.nan,
        "negative_diff_p01": float(np.nanquantile(neg_values, 0.01)) if neg_values.size else np.nan,
        "negative_diff_p50": float(np.nanquantile(neg_values, 0.50)) if neg_values.size else np.nan,
        "negative_diff_p99": float(np.nanquantile(neg_values, 0.99)) if neg_values.size else np.nan,
        "precip_tiny_negative_tolerance_mm": float(PRECIP_TINY_NEGATIVE_TOLERANCE_MM),
        "precip_large_negative_policy": "set_to_nan_and_audit",
    }

    if np.count_nonzero(large_negative):
        prev_acc = np.empty_like(hourly, dtype="float32")
        prev_acc[0, ...] = np.nan
        prev_acc[1:, ...] = acc[:-1, ...]
        reset_like = (
            large_negative & np.isfinite(prev_acc) & (prev_acc > 1.0)
            & (hourly <= -PRECIP_RESET_DROP_THRESHOLD_MM)
        )
        diag["reset_like_count"] = int(np.count_nonzero(reset_like))

    # Clip tiny (<0.01 mm) accumulated-precipitation packing artefacts to zero;
    # treat larger drops as missing and expose them in the audit.
    hourly[tiny_negative] = 0.0
    if diag["large_negative_count"]:
        hourly[large_negative] = np.nan
    return hourly.astype("float32"), diag


def read_time_index_utc(ds: netCDF4.Dataset, expected_len: int | None = None) -> pd.DatetimeIndex:
    if "time" not in ds.variables:
        raise_parse_error(
            "empty_time_index",
            "NetCDF file has no time variable.",
            error_type="missing_time_variable",
        )
    time_var = ds.variables["time"]
    values = np.asarray(time_var[:], dtype="float64")
    units = getattr(time_var, "units", "")
    if values.size == 0:
        raise_parse_error("empty_time_index", "Time variable has length zero.")
    if expected_len is not None and values.size != expected_len:
        raise_parse_error(
            "time_length_mismatch",
            f"Time variable length {values.size} does not match target time axis {expected_len}.",
        )
    invalid_time_values = (~np.isfinite(values)) | (np.abs(values) > 1e20)
    if bool(np.any(invalid_time_values)):
        n_bad = int(np.count_nonzero(invalid_time_values))
        raise_parse_error(
            "invalid_time_index",
            f"Time variable contains {n_bad} invalid/sentinel value(s).",
            error_type="invalid_time_values",
        )
    if not units:
        raise_parse_error(
            "invalid_time_index",
            "Time variable has empty units attribute.",
            error_type="missing_time_units",
        )
    try:
        if units.lower().startswith("seconds since 1970-01-01"):
            decoded = pd.to_datetime(values, unit="s", utc=True)
        else:
            decoded_raw = netCDF4.num2date(
                values,
                units=units,
                calendar=getattr(time_var, "calendar", "standard"),
                only_use_cftime_datetimes=False,
                only_use_python_datetimes=True,
            )
            decoded = pd.to_datetime([pd.Timestamp(x) for x in decoded_raw], utc=True)
    except Exception as exc:
        raise_parse_error(
            "invalid_time_index",
            f"Could not decode time variable: {exc}",
            error_type=type(exc).__name__,
        )
    decoded = pd.DatetimeIndex(decoded)
    if len(decoded) == 0:
        raise_parse_error("empty_time_index", "Decoded time index is empty.")
    return decoded


def normalise_to_time_member_store(
    raw: np.ndarray,
    dims: tuple[str, ...],
    expected_time_len: int | None = None,
) -> np.ndarray:
    arr = np.asarray(raw, dtype="float32")
    dims = list(dims)
    if arr.ndim != len(dims):
        raise_parse_error("invalid_shape", f"Array ndim {arr.ndim} does not match NetCDF dimensions {dims}.")
    if arr.size == 0 or any(size == 0 for size in arr.shape):
        raise_parse_error(
            "empty_variable",
            f"Variable has zero-length dimension: shape {arr.shape}, dims {dims}.",
        )
    if "time" not in dims:
        raise_parse_error("invalid_shape", f"Expected a time dimension; found dimensions {dims}.")
    if "x" not in dims:
        raise_parse_error("invalid_shape", f"Expected an x/store dimension; found dimensions {dims}.")
    for dim in list(dims):
        if dim not in {"time", "ensemble_member", "x"}:
            axis = dims.index(dim)
            if arr.shape[axis] != 1:
                raise_parse_error(
                    "invalid_shape",
                    f"Unexpected non-singleton dimension {dim} with shape {arr.shape}.",
                )
            arr = np.take(arr, 0, axis=axis)
            dims.pop(axis)
    if "ensemble_member" not in dims:
        arr = np.expand_dims(arr, axis=1)
        dims.insert(1, "ensemble_member")
    order = [dims.index(dim) for dim in ["time", "ensemble_member", "x"]]
    arr = np.transpose(arr, axes=order)
    if expected_time_len is not None and arr.shape[0] != expected_time_len:
        raise_parse_error(
            "time_length_mismatch",
            f"Variable time axis {arr.shape[0]} does not match decoded time index "
            f"{expected_time_len}.",
        )
    if arr.shape[0] == 0:
        raise_parse_error(
            "empty_variable",
            f"Variable has zero-length time axis after normalisation: shape {arr.shape}.",
        )
    if arr.shape[1] == 0:
        raise_parse_error(
            "invalid_shape",
            f"Variable has zero ensemble members after normalisation: shape {arr.shape}.",
        )
    if arr.shape[2] != len(AVDELING_BY_X):
        raise_parse_error(
            "invalid_shape",
            f"Expected {len(AVDELING_BY_X)} stores on x dimension; "
            f"found {arr.shape[2]}.",
        )
    if not np.isfinite(arr).any():
        raise_parse_error(
            "all_missing_values",
            "Variable contains no finite usable values after fill-value masking.",
        )
    return arr.astype("float32")


def read_ensemble_array(
    ds: netCDF4.Dataset,
    ncvar: str,
    variable: str,
    expected_time_len: int | None = None,
) -> tuple[np.ndarray, dict]:
    if ncvar not in ds.variables:
        raise_parse_error(
            "missing_target_variable",
            f"Variable {ncvar!r} not found. Available variables: {list(ds.variables)}",
        )
    var = ds.variables[ncvar]
    attrs = {
        "units": getattr(var, "units", ""),
        "standard_name": getattr(var, "standard_name", ""),
        "long_name": getattr(var, "long_name", ""),
        "raw_shape": tuple(var.shape),
        "raw_dimensions": tuple(var.dimensions),
    }
    arr = normalise_to_time_member_store(
        mask_fill_values(var[:]),
        var.dimensions,
        expected_time_len=expected_time_len,
    )
    attrs["shape"] = arr.shape
    units = attrs["units"]
    if variable == "temp":
        arr = convert_temperature(arr, units)
    elif variable in {"humid", "cloud"}:
        arr = convert_fraction_to_percent(arr, units)
    elif variable == "precip":
        arr, precip_diag = convert_precip_accumulated_to_hourly(arr)
        attrs.update(precip_diag)
    return arr.astype("float32"), attrs


def aggregate_member_window(values: np.ndarray, variable: str) -> np.ndarray:
    any_finite = np.isfinite(values).any(axis=0)
    out = np.full(values.shape[1], np.nan, dtype="float32")
    if variable == "precip":
        out[any_finite] = np.nansum(values[:, any_finite], axis=0).astype("float32")
    else:
        with np.errstate(all="ignore"):
            out[any_finite] = np.nanmean(values[:, any_finite], axis=0).astype("float32")
    return out


def summarise_member_distribution(member_values: np.ndarray, prefix: str, variable: str) -> dict:
    finite = member_values[np.isfinite(member_values)]
    out = {f"{prefix}_ens_{stat}": np.nan for stat in ["mean", "std", "p10", "p50", "p90", "min", "max"]}
    if finite.size:
        out.update({
            f"{prefix}_ens_mean": float(np.nanmean(finite)),
            f"{prefix}_ens_std": float(np.nanstd(finite, ddof=1)) if finite.size > 1 else 0.0,
            f"{prefix}_ens_p10": float(np.nanpercentile(finite, 10)),
            f"{prefix}_ens_p50": float(np.nanpercentile(finite, 50)),
            f"{prefix}_ens_p90": float(np.nanpercentile(finite, 90)),
            f"{prefix}_ens_min": float(np.nanmin(finite)),
            f"{prefix}_ens_max": float(np.nanmax(finite)),
        })
    if variable == "precip":
        out["precip_ens_wet_prob"] = float(np.nanmean(finite >= WET_THRESHOLD_MM)) if finite.size else np.nan
    return out


def aggregate_array_to_windows(
    arr: np.ndarray,
    target_datetimes_utc: pd.DatetimeIndex,
    origin_datetime_utc,
    variable: str,
    prefix: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    arr = np.asarray(arr, dtype="float32")
    if arr.ndim != 3:
        raise_parse_error("invalid_shape", f"Expected array as (time, member, store); got shape {arr.shape}.")
    target_datetimes_utc = pd.DatetimeIndex(target_datetimes_utc).tz_convert("UTC")
    if len(target_datetimes_utc) == 0:
        raise_parse_error("empty_time_index", "Cannot aggregate because target time index is empty.")
    if arr.shape[0] != len(target_datetimes_utc):
        raise_parse_error(
            "time_length_mismatch",
            f"Array time axis {arr.shape[0]} does not match decoded time index "
            f"{len(target_datetimes_utc)}.",
        )
    if arr.shape[1] == 0 or arr.shape[2] == 0:
        raise_parse_error("invalid_shape", f"Array has empty member/store axis: shape {arr.shape}.")
    target_local = target_datetimes_utc.tz_convert(LOCAL_TZ)
    origin_datetime_utc = (
        pd.Timestamp(origin_datetime_utc, tz="UTC")
        if pd.Timestamp(origin_datetime_utc).tzinfo is None
        else pd.Timestamp(origin_datetime_utc).tz_convert("UTC")
    )
    origin_local_date = origin_datetime_utc.tz_convert(LOCAL_TZ).date()
    origin_date = origin_datetime_utc.tz_localize(None).normalize()
    origin_hour = np.int8(origin_datetime_utc.hour)

    meta = pd.DataFrame({
        "time_idx": np.arange(len(target_datetimes_utc), dtype="int16"),
        "target_date": target_local.normalize().tz_localize(None),
        "target_hour_local": target_local.hour.astype("int8"),
        "target_hour_utc": target_datetimes_utc.hour.astype("int8"),
    })
    meta["horizon_days"] = meta["target_date"].apply(
        lambda x: (pd.Timestamp(x).date() - origin_local_date).days
    ).astype("int8")
    meta = meta.loc[meta["horizon_days"].isin(WINDOW_OUTPUT_HORIZONS)].reset_index(drop=True)

    rows = []
    audit_rows = []
    expected_members = int(arr.shape[1])
    for (target_date, horizon_days), day_meta in meta.groupby(["target_date", "horizon_days"], observed=True):
        for window_name in WINDOW_ORDER:
            window = WINDOW_DEFINITIONS[window_name]
            sub_meta = day_meta.loc[
                day_meta["target_hour_local"].between(
                    window["start_hour"], window["end_hour"]
                )
            ]
            if sub_meta.empty:
                continue
            time_idx = sub_meta["time_idx"].to_numpy(dtype="int64")
            expected_hours = compute_expected_hours_for_window(target_date, window_name)
            for store_pos, avdeling_id in enumerate(AVDELING_BY_X):
                values = arr[time_idx, :, store_pos]
                hours_in_window = int(np.count_nonzero(np.isfinite(values).any(axis=1)))
                if hours_in_window == 0:
                    continue
                member_values = aggregate_member_window(values, variable)
                n_members_available = int(np.count_nonzero(np.isfinite(member_values)))
                missing_members = int(expected_members - n_members_available)
                base = {
                    "origin_date": origin_date,
                    "origin_hour": origin_hour,
                    "origin_datetime_utc": origin_datetime_utc.tz_localize(None),
                    "target_date": pd.Timestamp(target_date).normalize(),
                    "horizon_days": np.int8(horizon_days),
                    "AvdelingID": np.int64(avdeling_id),
                    "aggregation_window": window_name,
                    "hours_in_window": np.int8(hours_in_window),
                    "expected_hours_in_window": np.int8(expected_hours),
                    "coverage_share": np.float32(hours_in_window / expected_hours),
                    "is_partial_window": np.int8(hours_in_window != expected_hours),
                    "is_full_window": np.int8(hours_in_window == expected_hours),
                    "n_members_available": np.int16(n_members_available),
                }
                rows.append({
                    **base,
                    **summarise_member_distribution(member_values, prefix, variable),
                })
                audit_rows.append({
                    "audit_level": "window_detail",
                    "variable": variable,
                    "status": "parsed_ok",
                    **{key: base[key] for key in FINAL_KEY_COLS},
                    "hours_in_window": base["hours_in_window"],
                    "expected_hours_in_window": base["expected_hours_in_window"],
                    "coverage_share": base["coverage_share"],
                    "n_members_available": base["n_members_available"],
                    "missing_members": np.int16(missing_members),
                    "is_complete_variable": np.int8(missing_members == 0),
                    "is_complete_window": np.int8(
                        (hours_in_window == expected_hours) and (missing_members == 0)
                    ),
                })
    return pd.DataFrame(rows), pd.DataFrame(audit_rows)


def parse_scalar_file_to_windows(row: pd.Series, variable: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    cfg = ENSEMBLE_COMPONENTS[variable]
    path = Path(row["path"])
    if not path.exists():
        raise_parse_error(
            "no_files_available",
            f"NetCDF file does not exist: {path}",
            error_type="file_missing",
        )
    with netCDF4.Dataset(path) as ds:
        if cfg["ncvar"] not in ds.variables:
            raise_parse_error(
                "missing_target_variable",
                f"Variable {cfg['ncvar']!r} not found in {path.name}.",
            )
        raw_shape = tuple(ds.variables[cfg["ncvar"]].shape)
        raw_dims = tuple(ds.variables[cfg["ncvar"]].dimensions)
        time_len = raw_shape[raw_dims.index("time")] if "time" in raw_dims else None
        target_datetimes_utc = read_time_index_utc(ds, expected_len=time_len)
        arr, attrs = read_ensemble_array(
            ds, cfg["ncvar"], variable, expected_time_len=len(target_datetimes_utc)
        )
    parsed, coverage = aggregate_array_to_windows(
        arr,
        target_datetimes_utc,
        pd.Timestamp(row["origin_datetime_utc"], tz="UTC"),
        variable,
        cfg["prefix"],
    )
    if variable == "precip":
        for col in PRECIP_AUDIT_COLUMNS:
            coverage[col] = attrs.get(col, np.nan)
        if attrs.get("large_negative_count", 0):
            print(
                f"WARNING: {Path(row['path']).name} had {attrs['large_negative_count']} "
                "large negative accumulated-precip diffs set to NaN; see coverage audit."
            )
    return parsed, coverage


def parse_wind_pair_to_windows(row: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
    path_x = Path(row["path_x"])
    path_y = Path(row["path_y"])
    if not path_x.exists() or not path_y.exists():
        raise_parse_error(
            "wind_unpaired",
            f"Missing wind component file(s): x={path_x.exists()}, y={path_y.exists()}.",
            error_type="file_missing",
        )
    with netCDF4.Dataset(path_x) as ds_x:
        ncvar_x = ENSEMBLE_COMPONENTS["wind_x"]["ncvar"]
        if ncvar_x not in ds_x.variables:
            raise_parse_error(
                "missing_target_variable",
                f"Variable {ncvar_x!r} not found in {path_x.name}.",
            )
        raw_shape_x = tuple(ds_x.variables[ncvar_x].shape)
        raw_dims_x = tuple(ds_x.variables[ncvar_x].dimensions)
        time_len_x = raw_shape_x[raw_dims_x.index("time")] if "time" in raw_dims_x else None
        time_x = read_time_index_utc(ds_x, expected_len=time_len_x)
        arr_x, _ = read_ensemble_array(
            ds_x, ncvar_x, "wind_x", expected_time_len=len(time_x)
        )
    with netCDF4.Dataset(path_y) as ds_y:
        ncvar_y = ENSEMBLE_COMPONENTS["wind_y"]["ncvar"]
        if ncvar_y not in ds_y.variables:
            raise_parse_error(
                "missing_target_variable",
                f"Variable {ncvar_y!r} not found in {path_y.name}.",
            )
        raw_shape_y = tuple(ds_y.variables[ncvar_y].shape)
        raw_dims_y = tuple(ds_y.variables[ncvar_y].dimensions)
        time_len_y = raw_shape_y[raw_dims_y.index("time")] if "time" in raw_dims_y else None
        time_y = read_time_index_utc(ds_y, expected_len=time_len_y)
        arr_y, _ = read_ensemble_array(
            ds_y, ncvar_y, "wind_y", expected_time_len=len(time_y)
        )
    if len(time_x) != len(time_y) or not np.array_equal(time_x.view("int64"), time_y.view("int64")):
        raise_parse_error(
            "wind_time_mismatch",
            f"Wind x/y time mismatch for origin {row['origin_datetime_utc']}.",
        )
    if arr_x.shape != arr_y.shape:
        raise_parse_error(
            "wind_shape_mismatch",
            f"Wind x/y shape mismatch for origin {row['origin_datetime_utc']}: "
            f"{arr_x.shape} vs {arr_y.shape}.",
        )
    wind_speed = np.sqrt(
        np.square(arr_x, dtype="float32") + np.square(arr_y, dtype="float32")
    ).astype("float32")
    return aggregate_array_to_windows(
        wind_speed,
        time_x,
        pd.Timestamp(row["origin_datetime_utc"], tz="UTC"),
        "wind",
        "wind",
    )


def summary_columns_for_variable(variable: str) -> list[str]:
    cols = [
        f"{variable}_ens_{stat}"
        for stat in ["mean", "std", "p10", "p50", "p90", "min", "max"]
    ]
    if variable == "precip":
        cols.append("precip_ens_wet_prob")
    return cols


## Optional smoke test

The smoke test parses a small sample for each available variable while keeping the full parser disabled. It is intended to detect schema, wind-pairing, and unit-conversion problems before a long run.


In [ ]:
smoke_variable_frames = {}
smoke_coverage_frames = []

if not RUN_SMOKE_TEST:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True to parse a tiny sample.")
else:
    print(f"Running smoke test with up to {SMOKE_FILES_PER_VARIABLE} file(s) per variable.")
    for variable in SCALAR_VARIABLES:
        index = file_indexes.get(variable, pd.DataFrame())
        if index.empty:
            print(f"Smoke test: {variable} skipped because no files were indexed.")
            continue
        frames, audits = [], []
        for _, row in index.head(SMOKE_FILES_PER_VARIABLE).iterrows():
            parsed, audit = parse_scalar_file_to_windows(row, variable)
            frames.append(parsed)
            audits.append(audit)
        smoke_variable_frames[variable] = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        smoke_coverage_frames.extend(audits)
        print(f"Smoke test: {variable} rows={len(smoke_variable_frames[variable])}")

    if wind_pairs.empty:
        print("Smoke test: wind skipped because no paired wind_x/wind_y origins are available.")
    else:
        frames, audits = [], []
        for _, row in wind_pairs.head(SMOKE_FILES_PER_VARIABLE).iterrows():
            parsed, audit = parse_wind_pair_to_windows(row)
            frames.append(parsed)
            audits.append(audit)
        smoke_variable_frames["wind"] = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        smoke_coverage_frames.extend(audits)
        print(f"Smoke test: wind rows={len(smoke_variable_frames['wind'])}")

    if smoke_coverage_frames:
        smoke_coverage = pd.concat(smoke_coverage_frames, ignore_index=True)
        display(
            smoke_coverage.groupby(["variable", "aggregation_window"], observed=True)
            .agg(
                rows=("variable", "size"),
                mean_coverage=("coverage_share", "mean"),
                min_members=("n_members_available", "min"),
                max_members=("n_members_available", "max"),
            )
            .reset_index()
        )


## Full ensemble parsing gate

This cell controls whether full ensemble parsing is allowed. With `RUN_FULL_PARSE = False`, the notebook can still produce availability audits without constructing the final daily-window ensemble output.


In [ ]:
def parse_variable_collection(variable: str, limit: int | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    if variable == "wind":
        index = wind_pairs.copy()
        if index.empty:
            print("wind: no paired wind_x/wind_y files available; wind ensemble is unavailable.")
            audit = build_file_parse_audit_row(
                None,
                "wind",
                status="wind_unpaired",
                error_type="wind_unpaired",
                notes="No paired wind_x/wind_y origins are available.",
            )
            return pd.DataFrame(), audit
        parser = parse_wind_pair_to_windows
    else:
        index = file_indexes.get(variable, pd.DataFrame()).copy()
        if index.empty:
            print(f"{variable}: no files available.")
            audit = build_file_parse_audit_row(
                None,
                variable,
                status="no_files_available",
                error_type="no_files_available",
                notes=f"No files indexed for {variable}.",
            )
            return pd.DataFrame(), audit
        parser = lambda row: parse_scalar_file_to_windows(row, variable)
    if limit is not None:
        index = index.head(limit).copy()

    parsed_frames, audit_frames = [], []
    status_counts = {}
    start = time.time()
    for pos, (_, row) in enumerate(
        tqdm(index.iterrows(), total=len(index), desc=f"parse {variable}"),
        start=1,
    ):
        try:
            parsed, audit = parser(row)
            if not parsed.empty:
                parsed_frames.append(parsed)
            if not audit.empty:
                audit_frames.append(audit)
            audit_frames.append(build_file_success_audit_row(row, variable, parsed, audit))
            status = "parsed_ok"
        except NetCDFParseError as exc:
            audit_frames.append(
                build_file_parse_audit_row(
                    row,
                    variable,
                    status=exc.status,
                    error_type=exc.error_type,
                    error_message=str(exc),
                )
            )
            status = exc.status
            path_for_log = _series_value(row, "path_x" if variable == "wind" else "path", "")
            print(
                f"{variable}: skipped {pos:,}/{len(index):,} "
                f"{Path(path_for_log).name}: {exc.status} - {exc}"
            )
        except Exception as exc:
            audit_frames.append(
                build_file_parse_audit_row(
                    row,
                    variable,
                    status="parse_error",
                    error_type=type(exc).__name__,
                    error_message=str(exc),
                )
            )
            status = "parse_error"
            print(
                f"{variable}: skipped {pos:,}/{len(index):,} due to parse_error "
                f"{type(exc).__name__}: {exc}"
            )
        status_counts[status] = status_counts.get(status, 0) + 1
        if pos % 100 == 0:
            print(
                f"{variable}: processed {pos:,}/{len(index):,} files in "
                f"{(time.time() - start) / 60:.1f} min; statuses={status_counts}"
            )
            gc.collect()
    print(f"{variable}: parse status summary {status_counts}")
    return (
        pd.concat(parsed_frames, ignore_index=True) if parsed_frames else pd.DataFrame(),
        pd.concat(audit_frames, ignore_index=True, sort=False) if audit_frames else pd.DataFrame(),
    )


variable_frames = {}
coverage_frames = []

if not RUN_FULL_PARSE:
    print("Full ensemble parsing skipped. Set RUN_FULL_PARSE = True in Cell 1 to run later.")
else:
    INTERIM_DIR.mkdir(parents=True, exist_ok=True)
    for variable in FINAL_VARIABLES:
        frame, coverage = parse_variable_collection(variable)
        if not coverage.empty:
            coverage_frames.append(coverage)
        if frame.empty:
            print(f"{variable}: no parsed rows; checkpoint not written.")
            continue
        variable_frames[variable] = frame
        value_path = INTERIM_DIR / f"forecast_meps_ensemble_{variable}_daily_windows.parquet"
        coverage_path = INTERIM_DIR / f"forecast_meps_ensemble_{variable}_coverage_audit.parquet"
        frame.to_parquet(value_path, index=False, engine="pyarrow", compression="snappy")
        coverage.to_parquet(coverage_path, index=False, engine="pyarrow", compression="snappy")
        print(f"{variable}: wrote checkpoint {value_path} rows={len(frame):,}")
        del frame, coverage
        gc.collect()


## Variable checkpoints and window aggregates

The notebook loads variable-level checkpoints and prepares the per-window ensemble summaries used for the merged daily-window output. This keeps each weather variable separable for coverage and quality checks.


In [ ]:
def load_variable_checkpoints() -> tuple[dict[str, pd.DataFrame], list[pd.DataFrame]]:
    loaded_variables = {}
    loaded_coverage = []
    for variable in FINAL_VARIABLES:
        value_path = INTERIM_DIR / f"forecast_meps_ensemble_{variable}_daily_windows.parquet"
        coverage_path = INTERIM_DIR / f"forecast_meps_ensemble_{variable}_coverage_audit.parquet"
        if value_path.exists():
            loaded_variables[variable] = pd.read_parquet(value_path)
            print(f"Loaded {variable} checkpoint rows={len(loaded_variables[variable]):,}")
        else:
            print(f"No {variable} value checkpoint found at {value_path}")
        if coverage_path.exists():
            loaded_coverage.append(pd.read_parquet(coverage_path))
    return loaded_variables, loaded_coverage


if RUN_FULL_PARSE and not variable_frames:
    variable_frames, coverage_frames = load_variable_checkpoints()

if RUN_FULL_PARSE and coverage_frames:
    coverage_preview = pd.concat(coverage_frames, ignore_index=True)
    display(
        coverage_preview.groupby(["variable", "aggregation_window"], observed=True)
        .agg(
            rows=("variable", "size"),
            mean_coverage=("coverage_share", "mean"),
            min_coverage=("coverage_share", "min"),
            mean_members=("n_members_available", "mean"),
            min_members=("n_members_available", "min"),
        )
        .reset_index()
    )
else:
    print("No full-parse coverage frames loaded. This is expected while RUN_FULL_PARSE = False.")


## Variable merge

The available ensemble-variable outputs are merged into one daily-window panel keyed by forecast origin, target date, horizon, store, and aggregation window. Partial-output handling remains explicit and gated.


In [ ]:

def _format_origin_hour_values(index: pd.DataFrame) -> str:
    if index is None or index.empty or "origin_hour" not in index.columns:
        return ""
    values = sorted(pd.Series(index["origin_hour"]).dropna().astype(int).unique().tolist())
    return ",".join(str(v) for v in values)


def _origin_date_bounds(index: pd.DataFrame) -> tuple[object, object]:
    if index is None or index.empty or "origin_date" not in index.columns:
        return pd.NaT, pd.NaT
    return pd.to_datetime(index["origin_date"]).min(), pd.to_datetime(index["origin_date"]).max()


def _folder_row(component: str) -> pd.Series:
    rows = folder_audit.loc[folder_audit["component"] == component]
    if rows.empty:
        cfg = ENSEMBLE_COMPONENTS[component]
        return pd.Series({
            "component": component,
            "folder": cfg["folder"],
            "path": str(RAW_DATA_ROOT / cfg["folder"]),
            "exists": False,
            "file_count": 0,
            "parsed_file_count": 0,
            "bad_filename_count": 0,
        })
    return rows.iloc[0]


def _scalar_availability_row(variable: str, expected_files: int) -> dict:
    component = variable
    folder = _folder_row(component)
    index = file_indexes.get(component, pd.DataFrame())
    files_available = int(folder.get("file_count", 0) or 0)
    parsed_file_count = int(folder.get("parsed_file_count", 0) or 0)
    bad_filename_count = int(folder.get("bad_filename_count", 0) or 0)
    origin_min, origin_max = _origin_date_bounds(index)
    share = (files_available / expected_files) if expected_files else np.nan
    notes = []
    if not bool(folder.get("exists", False)):
        notes.append("source folder missing")
    elif files_available == 0:
        notes.append("no NetCDF files available")
    if bad_filename_count:
        notes.append(f"{bad_filename_count} file(s) did not match filename pattern")
    if files_available and expected_files and files_available < expected_files:
        notes.append("folder has fewer files than the largest ensemble component folder")
    return {
        "audit_level": "folder_availability",
        "variable": variable,
        "source_folder": str(folder.get("folder", ENSEMBLE_COMPONENTS[component]["folder"])),
        "path": str(folder.get("path", RAW_DATA_ROOT / ENSEMBLE_COMPONENTS[component]["folder"])),
        "status": "folder_available" if parsed_file_count > 0 else "no_files_available",
        "error_type": "",
        "error_message": "",
        "files_available": files_available,
        "expected_files": expected_files if expected_files else pd.NA,
        "file_availability_share": np.float32(share) if pd.notna(share) else np.nan,
        "origin_date_min": origin_min,
        "origin_date_max": origin_max,
        "origin_hour_values": _format_origin_hour_values(index),
        "n_origins": (
            int(index["origin_datetime_utc"].nunique())
            if not index.empty and "origin_datetime_utc" in index.columns else 0
        ),
        "n_paired_origins": pd.NA,
        "n_unpaired_wind_x": pd.NA,
        "n_unpaired_wind_y": pd.NA,
        "wind_x_files_available": pd.NA,
        "wind_y_files_available": pd.NA,
        "aggregation_window": pd.NA,
        "horizon_days": pd.NA,
        "lead_hour_coverage": pd.NA,
        "n_rows_available": parsed_file_count,
        "coverage_share": np.float32(share) if pd.notna(share) else np.nan,
        "is_available": int(parsed_file_count > 0),
        "is_partial": int(
            bool(expected_files and files_available < expected_files)
            or bad_filename_count > 0
            or parsed_file_count == 0
        ),
        "notes": "; ".join(notes) if notes else "available for independent scalar parsing",
    }


def _wind_availability_row(expected_files: int) -> dict:
    folder_x = _folder_row("wind_x")
    folder_y = _folder_row("wind_y")
    index_x = file_indexes.get("wind_x", pd.DataFrame())
    index_y = file_indexes.get("wind_y", pd.DataFrame())
    set_x = (
        set(pd.to_datetime(index_x["origin_datetime_utc"]).astype(str).tolist())
        if not index_x.empty else set()
    )
    set_y = (
        set(pd.to_datetime(index_y["origin_datetime_utc"]).astype(str).tolist())
        if not index_y.empty else set()
    )
    paired = int(len(wind_pairs)) if isinstance(wind_pairs, pd.DataFrame) else 0
    unpaired_x = int(len(set_x - set_y))
    unpaired_y = int(len(set_y - set_x))
    origin_parts = [idx for idx in [index_x, index_y] if idx is not None and not idx.empty]
    origin_union = pd.concat(origin_parts, ignore_index=True) if origin_parts else pd.DataFrame()
    origin_min, origin_max = _origin_date_bounds(origin_union)
    files_x = int(folder_x.get("file_count", 0) or 0)
    files_y = int(folder_y.get("file_count", 0) or 0)
    share = (paired / expected_files) if expected_files else np.nan
    notes = []
    if files_x == 0:
        notes.append("wind_x unavailable")
    if files_y == 0:
        notes.append("wind_y unavailable; wind speed ensemble skipped")
    if unpaired_x or unpaired_y:
        notes.append("wind speed uses inner-paired wind_x/wind_y origins only")
    if paired == 0:
        notes.append("no paired wind origins available")
    return {
        "audit_level": "folder_availability",
        "variable": "wind",
        "source_folder": (
            f"{ENSEMBLE_COMPONENTS['wind_x']['folder']} + "
            f"{ENSEMBLE_COMPONENTS['wind_y']['folder']}"
        ),
        "path": (
            f"{folder_x.get('path', RAW_DATA_ROOT / ENSEMBLE_COMPONENTS['wind_x']['folder'])} | "
            f"{folder_y.get('path', RAW_DATA_ROOT / ENSEMBLE_COMPONENTS['wind_y']['folder'])}"
        ),
        "status": "folder_available" if paired > 0 else "wind_unpaired",
        "error_type": "",
        "error_message": "",
        "files_available": paired,
        "expected_files": expected_files if expected_files else pd.NA,
        "file_availability_share": np.float32(share) if pd.notna(share) else np.nan,
        "origin_date_min": origin_min,
        "origin_date_max": origin_max,
        "origin_hour_values": _format_origin_hour_values(origin_union),
        "n_origins": int(max(len(set_x), len(set_y))),
        "n_paired_origins": paired,
        "n_unpaired_wind_x": unpaired_x,
        "n_unpaired_wind_y": unpaired_y,
        "wind_x_files_available": files_x,
        "wind_y_files_available": files_y,
        "aggregation_window": pd.NA,
        "horizon_days": pd.NA,
        "lead_hour_coverage": pd.NA,
        "n_rows_available": paired,
        "coverage_share": np.float32(share) if pd.notna(share) else np.nan,
        "is_available": int(paired > 0),
        "is_partial": int(
            paired == 0
            or bool(expected_files and paired < expected_files)
            or unpaired_x > 0
            or unpaired_y > 0
        ),
        "notes": "; ".join(notes) if notes else "wind_x and wind_y are fully paired for available origins",
    }


def build_folder_availability_audit() -> pd.DataFrame:
    expected_files = (
        int(folder_audit["file_count"].max())
        if not folder_audit.empty and folder_audit["file_count"].max() > 0 else 0
    )
    rows = [_scalar_availability_row(variable, expected_files) for variable in SCALAR_VARIABLES]
    rows.append(_wind_availability_row(expected_files))
    out = pd.DataFrame(rows)
    return out.sort_values(["variable", "audit_level"]).reset_index(drop=True)


def _folder_context_by_variable(folder_level_audit: pd.DataFrame) -> dict[str, dict]:
    if folder_level_audit.empty:
        return {}
    return folder_level_audit.drop_duplicates("variable").set_index("variable").to_dict("index")


def _concat_coverage_frames(coverage_frames: list[pd.DataFrame]) -> pd.DataFrame:
    usable = [frame for frame in coverage_frames if frame is not None and not frame.empty]
    return pd.concat(usable, ignore_index=True, sort=False) if usable else pd.DataFrame()


def build_file_parse_audit(coverage_frames: list[pd.DataFrame]) -> pd.DataFrame:
    detail = _concat_coverage_frames(coverage_frames)
    if detail.empty or "audit_level" not in detail.columns:
        return pd.DataFrame()
    return detail.loc[detail["audit_level"].eq("file_parse")].reset_index(drop=True)


def build_window_coverage_audit(
    coverage_frames: list[pd.DataFrame],
    folder_level_audit: pd.DataFrame,
) -> pd.DataFrame:
    detail = _concat_coverage_frames(coverage_frames)
    if detail.empty or "aggregation_window" not in detail.columns or "horizon_days" not in detail.columns:
        return pd.DataFrame()
    detail = detail.dropna(subset=["aggregation_window", "horizon_days"])
    if detail.empty:
        return pd.DataFrame()
    contexts = _folder_context_by_variable(folder_level_audit)
    rows = []
    for (variable, aggregation_window, horizon_days), sub in detail.groupby(
        ["variable", "aggregation_window", "horizon_days"], observed=True
    ):
        ctx = contexts.get(variable, {})
        coverage = pd.to_numeric(sub["coverage_share"], errors="coerce")
        members = (
            pd.to_numeric(sub["n_members_available"], errors="coerce")
            if "n_members_available" in sub.columns else pd.Series(dtype="float64")
        )
        rows.append({
            "audit_level": "window_coverage",
            "variable": variable,
            "source_folder": ctx.get("source_folder", pd.NA),
            "path": pd.NA,
            "status": "parsed_ok",
            "error_type": "",
            "error_message": "",
            "files_available": ctx.get("files_available", pd.NA),
            "expected_files": ctx.get("expected_files", pd.NA),
            "file_availability_share": ctx.get("file_availability_share", np.nan),
            "origin_date_min": (
                pd.to_datetime(sub["origin_date"]).min()
                if "origin_date" in sub.columns else pd.NaT
            ),
            "origin_date_max": (
                pd.to_datetime(sub["origin_date"]).max()
                if "origin_date" in sub.columns else pd.NaT
            ),
            "origin_hour_values": _format_origin_hour_values(sub),
            "n_origins": (
                int(sub["origin_datetime_utc"].nunique())
                if "origin_datetime_utc" in sub.columns else pd.NA
            ),
            "n_paired_origins": ctx.get("n_paired_origins", pd.NA),
            "n_unpaired_wind_x": ctx.get("n_unpaired_wind_x", pd.NA),
            "n_unpaired_wind_y": ctx.get("n_unpaired_wind_y", pd.NA),
            "wind_x_files_available": ctx.get("wind_x_files_available", pd.NA),
            "wind_y_files_available": ctx.get("wind_y_files_available", pd.NA),
            "aggregation_window": aggregation_window,
            "horizon_days": int(horizon_days),
            "lead_hour_coverage": pd.NA,
            "n_rows_available": int(len(sub)),
            "coverage_share": float(coverage.mean()) if len(coverage.dropna()) else np.nan,
            "min_coverage_share": float(coverage.min()) if len(coverage.dropna()) else np.nan,
            "p05_coverage_share": float(coverage.quantile(0.05)) if len(coverage.dropna()) else np.nan,
            "max_coverage_share": float(coverage.max()) if len(coverage.dropna()) else np.nan,
            "mean_members_available": float(members.mean()) if len(members.dropna()) else np.nan,
            "min_members_available": float(members.min()) if len(members.dropna()) else np.nan,
            "max_members_available": float(members.max()) if len(members.dropna()) else np.nan,
            "is_available": int(len(sub) > 0),
            "is_partial": int((coverage < 1.0).any()) if len(coverage.dropna()) else 1,
            "notes": "Parsed window coverage aggregated across stores and forecast origins.",
        })
    return (
        pd.DataFrame(rows)
        .sort_values(["variable", "aggregation_window", "horizon_days"])
        .reset_index(drop=True)
    )


def build_ensemble_coverage_audit(coverage_frames: list[pd.DataFrame]) -> pd.DataFrame:
    folder_level = build_folder_availability_audit()
    file_level = build_file_parse_audit(coverage_frames)
    window_level = build_window_coverage_audit(coverage_frames, folder_level)
    parts = [folder_level]
    if not file_level.empty:
        parts.append(file_level)
    if not window_level.empty:
        parts.append(window_level)
    audit = pd.concat(parts, ignore_index=True, sort=False)
    for col in PRECIP_AUDIT_COLUMNS:
        if col not in audit.columns:
            audit[col] = np.nan
    preferred = [
        "audit_level", "variable", "source_folder", "path", "files_available",
        "expected_files", "file_availability_share", "origin_date_min",
        "origin_date_max", "origin_date", "origin_hour", "origin_datetime_utc",
        "origin_hour_values", "n_origins", "n_paired_origins",
        "n_unpaired_wind_x", "n_unpaired_wind_y", "wind_x_files_available",
        "wind_y_files_available",
        "aggregation_window", "horizon_days", "lead_hour_coverage", "n_rows_available", "coverage_share",
        "min_coverage_share", "p05_coverage_share", "max_coverage_share", "mean_members_available",
        "min_members_available", "max_members_available", "is_available",
        "is_partial", "status", "error_type", "error_message", "notes",
    ]
    return audit[
        [col for col in preferred if col in audit.columns]
        + [col for col in audit.columns if col not in preferred]
    ]


def merge_variable_outputs(variable_frames: dict[str, pd.DataFrame]) -> pd.DataFrame | None:
    available = [v for v in FINAL_VARIABLES if v in variable_frames and not variable_frames[v].empty]
    missing = [v for v in FINAL_VARIABLES if v not in available]
    if missing:
        print(f"Missing variable outputs before merge: {missing}")
        if "wind" in missing:
            print("Wind is missing unless both wind_x and wind_y were paired and parsed.")
        if not ALLOW_PARTIAL_FINAL_OUTPUT:
            print("Final ensemble output blocked because ALLOW_PARTIAL_FINAL_OUTPUT = False.")
            return None

    merged = None
    variable_metadata_cols = []
    for variable in available:
        frame = variable_frames[variable].copy()
        keep_cols = (
            FINAL_KEY_COLS
            + summary_columns_for_variable(variable)
            + [
                "hours_in_window", "coverage_share", "is_full_window",
                "is_partial_window", "n_members_available",
            ]
        )
        rename = {
            "hours_in_window": f"{variable}_hours_in_window",
            "coverage_share": f"{variable}_coverage_share",
            "is_full_window": f"{variable}_is_full_window",
            "is_partial_window": f"{variable}_is_partial_window",
            "n_members_available": f"{variable}_n_members_available",
        }
        frame = frame[keep_cols].rename(columns=rename)
        variable_metadata_cols.extend(rename.values())
        merged = frame if merged is None else merged.merge(frame, on=FINAL_KEY_COLS, how="outer")

    if merged is None or merged.empty:
        return None
    for variable in FINAL_VARIABLES:
        if variable not in available and ALLOW_PARTIAL_FINAL_OUTPUT:
            for col in summary_columns_for_variable(variable):
                merged[col] = np.nan
    merged["expected_hours_in_window"] = assign_expected_hours_for_windows(merged, "target_date")
    hour_cols = [c for c in merged.columns if c.endswith("_hours_in_window")]
    member_cols = [c for c in merged.columns if c.endswith("_n_members_available")]
    merged["hours_in_window"] = merged[hour_cols].min(axis=1).astype("int8") if hour_cols else np.int8(0)
    merged["n_members_available"] = (
        merged[member_cols].min(axis=1).astype("int16")
        if member_cols else np.int16(0)
    )
    merged["coverage_share"] = (
        merged["hours_in_window"] / merged["expected_hours_in_window"]
    ).astype("float32")
    merged["is_full_window"] = (
        merged["hours_in_window"] == merged["expected_hours_in_window"]
    ).astype("int8")
    merged["is_partial_window"] = (1 - merged["is_full_window"]).astype("int8")
    merged = merged.drop(columns=[c for c in variable_metadata_cols if c in merged.columns])

    available_variables = ",".join(available)
    missing_variables = ",".join(missing)
    merged["is_partial_final_output"] = np.int8(len(missing) > 0)
    merged["available_variables"] = available_variables
    merged["missing_variables"] = missing_variables
    merged["ensemble_variable_coverage_summary"] = (
        f"available={available_variables or 'none'}; "
        f"missing={missing_variables or 'none'}"
    )
    for variable in FINAL_VARIABLES:
        merged[f"has_{variable}_ensemble"] = np.int8(variable in available)

    ordered = FINAL_KEY_COLS[:]
    for variable in FINAL_VARIABLES:
        ordered += [c for c in summary_columns_for_variable(variable) if c in merged.columns]
    ordered += [
        "hours_in_window", "expected_hours_in_window", "coverage_share",
        "is_partial_window", "is_full_window",
        "n_members_available", "is_partial_final_output", "available_variables", "missing_variables",
        "ensemble_variable_coverage_summary",
    ]
    ordered += [f"has_{variable}_ensemble" for variable in FINAL_VARIABLES]
    return (
        merged[ordered + [c for c in merged.columns if c not in ordered]]
        .sort_values(FINAL_KEY_COLS)
        .reset_index(drop=True)
    )


ensemble_windows = None
coverage_audit = build_ensemble_coverage_audit(coverage_frames)

if RUN_FULL_PARSE:
    ensemble_windows = merge_variable_outputs(variable_frames)
    if ensemble_windows is not None:
        print(f"Merged ensemble window rows: {len(ensemble_windows):,}")
        display(ensemble_windows.head())
else:
    print("Merge skipped because RUN_FULL_PARSE = False.")

if coverage_audit.empty:
    print("Coverage audit is empty. Check folder indexing before saving.")
else:
    print(f"Coverage audit rows ready to save: {len(coverage_audit):,}")
    display(
        coverage_audit.groupby(["audit_level", "variable"], observed=True)
        .agg(
            rows=("variable", "size"),
            available=("is_available", "max"),
            partial=("is_partial", "max"),
        )
        .reset_index()
    )


## Processed checkpoints

This step writes the ensemble coverage audit and, when allowed by the parse gates, the merged daily-window ensemble output. Canonical final outputs require complete variable coverage unless partial output is explicitly enabled.


In [ ]:

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if coverage_audit.empty:
    print("WARNING: coverage audit is empty; not writing coverage audit output.")
else:
    coverage_audit.to_parquet(
        ENSEMBLE_COVERAGE_AUDIT_OUTPUT_PATH,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )
    print(f"Wrote {ENSEMBLE_COVERAGE_AUDIT_OUTPUT_PATH} rows={len(coverage_audit):,}")

if not RUN_FULL_PARSE:
    print("Final ensemble daily-window output skipped because RUN_FULL_PARSE = False.")
elif ensemble_windows is None:
    print(
        "Final ensemble daily-window output skipped because required variables are "
        "missing or ALLOW_PARTIAL_FINAL_OUTPUT = False."
    )
else:
    ensemble_windows.to_parquet(
        ENSEMBLE_WINDOWS_OUTPUT_PATH,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )
    print(f"Wrote {ENSEMBLE_WINDOWS_OUTPUT_PATH} rows={len(ensemble_windows):,}")


## Sanity checks

The checks validate row uniqueness, horizon/window coverage, missing-variable flags, physical ranges, and partial-output status in the processed ensemble outputs.


In [ ]:

if not ENSEMBLE_COVERAGE_AUDIT_OUTPUT_PATH.exists():
    print(
        f"Coverage audit output not found at {ENSEMBLE_COVERAGE_AUDIT_OUTPUT_PATH}; "
        "audit checks skipped."
    )
else:
    audit_check = pd.read_parquet(ENSEMBLE_COVERAGE_AUDIT_OUTPUT_PATH)
    required_audit_cols = [
        "audit_level", "variable", "source_folder", "path", "files_available",
        "expected_files", "file_availability_share",
        "origin_date_min", "origin_date_max", "origin_hour_values", "n_origins", "aggregation_window",
        "horizon_days", "n_rows_available", "coverage_share", "is_available",
        "is_partial", "status", "error_type", "error_message", "notes",
    ]
    missing_audit_cols = [col for col in required_audit_cols if col not in audit_check.columns]
    if missing_audit_cols:
        raise AssertionError(f"Coverage audit missing required columns: {missing_audit_cols}")
    if audit_check.empty:
        raise AssertionError("Coverage audit output is empty.")
    if "wind" in set(audit_check["variable"]):
        wind_audit = audit_check.loc[audit_check["variable"] == "wind"]
        wind_required = [
            "wind_x_files_available", "wind_y_files_available",
            "n_paired_origins", "n_unpaired_wind_x", "n_unpaired_wind_y",
        ]
        missing_wind_cols = [col for col in wind_required if col not in wind_audit.columns]
        if missing_wind_cols:
            raise AssertionError(f"Wind audit missing pairing columns: {missing_wind_cols}")
    print("Coverage audit checks passed.")
    display(
        audit_check.groupby(["audit_level", "variable"], observed=True)
        .agg(
            rows=("variable", "size"),
            available=("is_available", "max"),
            partial=("is_partial", "max"),
        )
        .reset_index()
    )

if not ENSEMBLE_WINDOWS_OUTPUT_PATH.exists():
    print(
        f"Final ensemble window output not found at {ENSEMBLE_WINDOWS_OUTPUT_PATH}; "
        "sanity checks skipped."
    )
else:
    check = pd.read_parquet(ENSEMBLE_WINDOWS_OUTPUT_PATH)
    required_cols = FINAL_KEY_COLS + [
        "hours_in_window", "expected_hours_in_window", "coverage_share",
        "is_partial_window", "is_full_window", "n_members_available",
    ]
    missing_cols = [col for col in required_cols if col not in check.columns]
    if missing_cols:
        raise AssertionError(f"Missing required columns: {missing_cols}")
    if check["AvdelingID"].dtype != "int64":
        raise AssertionError(f"AvdelingID must be int64, got {check['AvdelingID'].dtype}")
    duplicate_count = int(check.duplicated(FINAL_KEY_COLS).sum())
    if duplicate_count:
        raise AssertionError(f"Duplicate final keys: {duplicate_count}")
    if not set(check["aggregation_window"].unique()).issubset(set(WINDOW_ORDER)):
        raise AssertionError("Unexpected aggregation_window values")
    if not set(check["horizon_days"].dropna().astype(int).unique()).issubset(
        set(WINDOW_OUTPUT_HORIZONS)
    ):
        raise AssertionError("Unexpected horizon_days values")
    if not check["coverage_share"].between(0.0, 1.0).all():
        raise AssertionError("coverage_share outside [0, 1]")
    if (check["hours_in_window"] > check["expected_hours_in_window"]).any():
        raise AssertionError("hours_in_window exceeds expected_hours_in_window")
    numeric = check.select_dtypes(include=[np.number])
    sentinel_cells = int((numeric.abs() > 1e30).sum().sum())
    if sentinel_cells:
        raise AssertionError(f"Found sentinel-like numeric cells > 1e30: {sentinel_cells}")
    for prefix in ["humid", "cloud"]:
        for col in [c for c in check.columns if c.startswith(f"{prefix}_ens_")]:
            finite = check[col].dropna()
            if not finite.between(0.0, 100.0).all():
                raise AssertionError(f"{col} outside [0, 100]")
    for col in [c for c in check.columns if c.startswith("precip_ens_") and c != "precip_ens_wet_prob"]:
        if (check[col].dropna() < 0.0).any():
            raise AssertionError(f"{col} contains negative precipitation")
    if (
        "precip_ens_wet_prob" in check.columns
        and not check["precip_ens_wet_prob"].dropna().between(0.0, 1.0).all()
    ):
        raise AssertionError("precip_ens_wet_prob outside [0, 1]")
    for col in ["is_partial_final_output", "available_variables", "missing_variables"]:
        if col not in check.columns:
            raise AssertionError(f"Missing partial-output metadata column: {col}")
    print("Final ensemble sanity checks passed.")
    display(
        check.groupby(["aggregation_window", "horizon_days"], observed=True)
        .agg(
            rows=("AvdelingID", "size"),
            mean_coverage=("coverage_share", "mean"),
            min_coverage=("coverage_share", "min"),
            mean_members=("n_members_available", "mean"),
        )
        .reset_index()
    )


## Ensemble mean versus deterministic forecast

This diagnostic compares ensemble means with deterministic MEPS forecasts by horizon and window. It is a consistency check for the parser and not an operational demand-forecasting result.


In [ ]:
if not ENSEMBLE_WINDOWS_OUTPUT_PATH.exists():
    print("Ensemble output missing; deterministic comparison skipped.")
elif not DETERMINISTIC_WINDOWS_PATH.exists():
    print(
        f"Deterministic forecast window file missing at {DETERMINISTIC_WINDOWS_PATH}; "
        "comparison skipped."
    )
else:
    ens = pd.read_parquet(ENSEMBLE_WINDOWS_OUTPUT_PATH)
    det = pd.read_parquet(DETERMINISTIC_WINDOWS_PATH)
    compare = det.merge(ens, on=FINAL_KEY_COLS, how="inner")
    print(f"Matched deterministic/ensemble rows: {len(compare):,}")
    pairs = [
        ("temp_fcst_mean", "temp_ens_mean"),
        ("precip_fcst_sum", "precip_ens_mean"),
        ("wind_fcst_mean", "wind_ens_mean"),
        ("humid_fcst_mean", "humid_ens_mean"),
        ("cloud_fcst_mean", "cloud_ens_mean"),
    ]
    rows = []
    for window in WINDOW_ORDER:
        for horizon in sorted(compare["horizon_days"].dropna().unique()):
            sub = compare[
                (compare["aggregation_window"] == window)
                & (compare["horizon_days"] == horizon)
            ]
            row = {"aggregation_window": window, "horizon_days": int(horizon), "n": len(sub)}
            for det_col, ens_col in pairs:
                if det_col in sub.columns and ens_col in sub.columns:
                    valid = sub[[det_col, ens_col]].dropna()
                    row[f"corr_{ens_col}"] = (
                        valid[det_col].corr(valid[ens_col])
                        if len(valid) >= 2 else np.nan
                    )
                    std_col = ens_col.replace("_mean", "_std")
                    if std_col in sub.columns:
                        row[f"mean_{std_col}"] = sub[std_col].mean()
            rows.append(row)
    display(pd.DataFrame(rows))
    if "precip_ens_wet_prob" in compare.columns:
        display(
            compare.groupby(["aggregation_window", "horizon_days"], observed=True)
            .agg(
                rows=("AvdelingID", "size"),
                mean_precip_wet_prob=("precip_ens_wet_prob", "mean"),
                det_wet_share=(
                    "precip_fcst_sum",
                    lambda s: float((s >= WET_THRESHOLD_MM).mean()),
                ),
            )
            .reset_index()
        )


## Optional calibration scaffold

Future calibration can merge `realised_weather_daily_windows.parquet`, `forecast_meps_daily_windows.parquet`, and `forecast_meps_ensemble_daily_windows.parquet` to compare deterministic forecast errors with ensemble spread by variable, horizon, season, and window. `data/processed/forecast_error_calibration_windows.parquet` should only be created after ensemble coverage is sufficient and the calibration design is explicitly approved.


In [ ]:
RUN_OPTIONAL_CALIBRATION_STUB = False

if not RUN_OPTIONAL_CALIBRATION_STUB:
    print("Calibration stub skipped. Enable only after 01c outputs exist and coverage has passed validation.")
else:
    required = [REALISED_WINDOWS_PATH, DETERMINISTIC_WINDOWS_PATH, ENSEMBLE_WINDOWS_OUTPUT_PATH]
    missing = [path for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Calibration inputs missing: {missing}")
    raise NotImplementedError(
        "Calibration output is intentionally not implemented in this scaffold. "
        "Validate ensemble coverage and approve the calibration design first."
    )


## Manual run notes

After ensemble downloads are complete, first run the setup, folder audit, store mapping, file indexing, helper definitions, and smoke test with `RUN_SMOKE_TEST = True` and `RUN_FULL_PARSE = False`. The availability audit can be written without creating the final daily-window ensemble output. For canonical parsing, set `RUN_FULL_PARSE = True`, keep `ALLOW_PARTIAL_FINAL_OUTPUT = False`, inspect variable checkpoints under `data/interim/`, then merge, save, sanity-check, and compare ensemble means with deterministic forecasts.

Expected outputs are `data/processed/forecast_meps_ensemble_coverage_audit.parquet` after folder indexing/save cells and `data/processed/forecast_meps_ensemble_daily_windows.parquet` after a valid full parse. Partial final outputs require explicit labelling through `ALLOW_PARTIAL_FINAL_OUTPUT = True` and include `is_partial_final_output`, available/missing variable lists, and `has_<variable>_ensemble` flags. Later documentation updates should record row counts, file sizes, coverage results, wind-component handling, uncertainty-feature schema, and the calibration role for M4.


In [ ]:
print("01c scaffold summary")
print(f"Raw root: {RAW_DATA_ROOT}")
print(f"Expected output: {ENSEMBLE_WINDOWS_OUTPUT_PATH}")
print(f"Expected coverage audit: {ENSEMBLE_COVERAGE_AUDIT_OUTPUT_PATH}")
print(
    f"RUN_SMOKE_TEST={RUN_SMOKE_TEST}, RUN_FULL_PARSE={RUN_FULL_PARSE}, "
    f"ALLOW_PARTIAL_FINAL_OUTPUT={ALLOW_PARTIAL_FINAL_OUTPUT}"
)
if "folder_audit" in globals():
    display(folder_audit[[
        "component", "exists", "file_count", "first_origin_date",
        "last_origin_date", "share_of_largest_folder",
    ]])
if "wind_pairs" in globals():
    print(f"Paired wind origins available: {len(wind_pairs)}")
    if len(wind_pairs) == 0:
        print(
            "Known blocker: wind speed ensemble requires paired wind_x and wind_y "
            "files; wind_y is currently missing or incomplete."
        )
